In [1]:
import numpy as np
import pandas as pd
import os
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchvision import models
from torchvision.models import EfficientNet_V2_S_Weights



In [2]:
# Reproducibility: reduces run-to-run variance so score moves predictably toward target.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



In [3]:
num_tta = 5



In [4]:
data_root = "/kaggle/input/cassava-leaf-disease-classification"
train_csv_path = os.path.join(data_root, "train.csv")
sample_sub_path = os.path.join(data_root, "sample_submission.csv")
train_image_dir = os.path.join(data_root, "train_images")
test_image_dir = os.path.join(data_root, "test_images")



In [5]:
test_df = pd.read_csv(sample_sub_path)
test_df.head()



,image_id,label
0,1234294272.jpg,4
1,1234332763.jpg,4
2,1234375577.jpg,4
3,1234555380.jpg,4
4,1234571117.jpg,4


In [6]:
# Use the originally defined base preprocessing at inference time (previously unused),
# which is important for stability and accuracy.
efficientnet_transforms = A.Compose(
    [
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.Resize(384, 384),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)



In [7]:
# Training augmentation: keeps the same overall approach (EfficientNetV2-S + CE),
# but enables learning a task-relevant head instead of random guessing.
train_transform = A.Compose(
    [
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.RandomResizedCrop(size=(384, 384), scale=(0.8, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.1, rotate_limit=15, p=0.6),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)



/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [8]:
# TTA: keep your function/logic, but use lighter, more label-preserving aug.
# Heavy CoarseDropout on cassava can be destructive; this change is aimed at improving accuracy toward target.
tta_transform = A.Compose(
    [
        A.Resize(384, 384),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.08, rotate_limit=10, p=0.6),
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.4),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)




In [9]:
class CassavaTrainDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.loc[idx, "image_id"]
        label = int(self.dataframe.loc[idx, "label"])
        img_path = os.path.join(self.image_dir, img_name)

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image at: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = self.transform(image=image)["image"]
        return image, label


class CassavaTestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.loc[idx, "image_id"]
        img_path = os.path.join(self.image_dir, img_name)

        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image at: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Return raw numpy for TTA function compatibility (it applies transforms itself).
        return image, img_name




In [10]:
def tta_predict_single_model(model, image, tta_transform, device, n_tta=5):
    model.eval()
    tta_predictions = []

    if image.shape[-1] != 3:
        raise ValueError("Image must have 3 channels (H, W, 3)")

    with torch.no_grad():
        for _ in range(n_tta):
            augmented = tta_transform(image=image)["image"]
            augmented = augmented.unsqueeze(0).to(device)

            output = model(augmented)
            probs = F.softmax(output, dim=1)
            tta_predictions.append(probs)

    avg_probs = torch.mean(torch.stack(tta_predictions), dim=0)
    return avg_probs




In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device



device(type='cuda')

In [12]:
# Model: keep EfficientNetV2-S backbone; train the 5-class head (and lightly fine-tune).
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = models.efficientnet_v2_s(weights=weights)

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(num_features, 5))
model = model.to(device)



Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:00<00:00, 92.4MB/s]


In [13]:
# Minimal training loop to make predictions non-random and move accuracy toward target.
# Keeps the same architecture family and standard CE objective.
train_df = pd.read_csv(train_csv_path)

# Stratified split for stable generalization without changing overall approach.
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, val_idx = next(sss.split(train_df["image_id"], train_df["label"]))
tr_df = train_df.iloc[train_idx].reset_index(drop=True)
va_df = train_df.iloc[val_idx].reset_index(drop=True)

train_ds = CassavaTrainDataset(tr_df, train_image_dir, transform=train_transform)
val_ds = CassavaTrainDataset(va_df, train_image_dir, transform=efficientnet_transforms)

train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

# Fine-tune: small LR, a few epochs to fit within time while improving accuracy substantially vs random.
# (Not "early stopping"; fixed epochs.)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


EPOCHS = (
    2  # minimal compute; increases score a lot from 0.12 baseline without heavy changes
)
for epoch in range(EPOCHS):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    n_tr = 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        tr_loss += loss.item() * bs
        tr_acc += accuracy_from_logits(logits.detach(), y) * bs
        n_tr += bs

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    n_va = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)

            bs = x.size(0)
            va_loss += loss.item() * bs
            va_acc += accuracy_from_logits(logits, y) * bs
            n_va += bs

    print(
        f"epoch={epoch+1}/{EPOCHS} "
        f"train_loss={tr_loss/n_tr:.4f} train_acc={tr_acc/n_tr:.4f} "
        f"val_loss={va_loss/n_va:.4f} val_acc={va_acc/n_va:.4f}"
    )




epoch=1/2 train_loss=0.5214 train_acc=0.8140 val_loss=0.3783 val_acc=0.8804
epoch=2/2 train_loss=0.3993 train_acc=0.8630 val_loss=0.3451 val_acc=0.8884


In [14]:
def identity_collate(batch):
    return batch


test_dataset = CassavaTestDataset(test_df, test_image_dir, transform=None)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=identity_collate,
)



In [15]:
# Inference with your TTA flow.
ensemble_predictions = []
image_names = []

with torch.no_grad():
    for batch in test_loader:
        image, img_name = batch[0]

        if isinstance(img_name, (list, tuple)):
            img_name = img_name[0]

        if isinstance(image, torch.Tensor):
            image = image.detach().cpu().numpy()

        probs = tta_predict_single_model(
            model, image, tta_transform, device, n_tta=num_tta
        )
        final_pred = int(probs.argmax(dim=1).cpu().item())

        ensemble_predictions.append(final_pred)
        image_names.append(str(img_name))

len(image_names), len(ensemble_predictions)



(2676, 2676)

In [16]:
# Strict alignment to sample_submission order/length.
pred_map = dict(zip(image_names, ensemble_predictions))
submission_df = test_df.copy()
submission_df["label"] = submission_df["image_id"].map(pred_map)

# Fallback (shouldn't trigger) to keep submission valid.
submission_df["label"] = submission_df["label"].fillna(0).astype(int)

assert submission_df.shape[0] == test_df.shape[0]
assert list(submission_df.columns) == ["image_id", "label"]

submission_df.to_csv("submission.csv", index=False)
print("Submission file saved as 'submission.csv'")
print(submission_df.head())

Submission file saved as 'submission.csv'
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
